# RetailPulse: Phase 1 – Data Ingestion & Cleaning Pipeline

## 1. Project Objective
This notebook demonstrates the **Phase 1 Data Layer** of the RetailPulse platform.
We ingest the 2-year UCI Online Retail II dataset, audit data quality, apply systematic cleaning rules, and persist high-performance Parquet artifacts.

### Cleaning Rules Applied:
1. **Column Normalization**: Standardize inconsistent column headers (`Customer ID` -> `CustomerID`, `UnitPrice` -> `Price`).
2. **Null Customer Filtering**: Remove records with missing `CustomerID` necessary for customer-level analytics.
3. **Cancellation Handling**: Identify and isolate cancellation invoices (invoices prefixed with 'C' and negative quantities).
4. **Non-Positive Quantity & Price Removal**: Ensure `Quantity > 0` and `Price > 0`.
5. **Deduplication**: Drop exact duplicate rows.
6. **Type Casting**: Enforce strict datetimes and integer customer keys.
7. **Metric Calculation**: Derive `TotalPrice = Quantity * Price`.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import json

# Ensure project root is on path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.utils.config import load_config
from src.utils.logger import get_logger
from src.data.load import load_raw_data
from src.data.clean import clean_retail_data

config = load_config(str(root_dir / 'config.yaml'))
print(f"RetailPulse Configuration Loaded. Version: {config['project']['version']}")

## 2. Load Raw Transaction Data
We load both sheets ('Year 2009-2010' and 'Year 2010-2011') from the raw Excel workbook.

In [ ]:
raw_df = load_raw_data()
print(f"Raw transactions shape: {raw_df.shape}")
raw_df.head()

## 3. Execute Cleaning Pipeline & Audit Rules

In [ ]:
clean_df, audit_report = clean_retail_data(raw_df, config=config, save_output=True)
print("Cleaning Audit Report:")
print(json.dumps(audit_report, indent=2))

## 4. Validate Cleaned Dataset Integrity

In [ ]:
print(f"Total Clean Rows: {len(clean_df):,}")
print(f"Missing values check:\n{clean_df.isna().sum()}")
print("\nSummary Statistics:")
clean_df[['Quantity', 'Price', 'TotalPrice']].describe().round(2)

## 5. Summary & Phase 1 Sign-Off
- Clean data successfully saved to `data/processed/online_retail_cleaned.parquet`.
- High-speed columnar format ready for Feature Engineering in Phase 2.